In [ ]:
#    // AgriVision - Copyright (C) 2026 <Shahd Kazan>
#    // Licensed under the GNU AGPL-3.0. See the LICENSE file.

In [ ]:
# Install MLflow in Kaggle Notebook

!pip install -q mlflow

# Verify installation
import mlflow
print("MLflow Version:", mlflow.__version__)

In [ ]:
# =============================================================
# Leaf Disease Instance Segmentation — YOLOv8 (FULL EXPERIMENTAL PIPELINE)
# Kaggle-ready Production Notebook
# =============================================================

# =============================================================
# 0. Setup
# =============================================================
import shutil
import os
import json
import gc
import random
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedShuffleSplit

import torch
import mlflow
import mlflow.pytorch



# =============================================================
# 1. Install dependencies
# =============================================================

!pip install -q ultralytics mlflow opencv-python-headless
from ultralytics import YOLO
from ultralytics import settings

settings.update({"mlflow": False})
import os
os.environ["YOLO_VERBOSE"] = "False"
import warnings
warnings.filterwarnings("ignore")
# =============================================================
# MLflow manual setup
# =============================================================

mlflow.set_tracking_uri("file:/kaggle/working/mlruns")
mlflow.set_experiment("LeafDiseaseSegmentation2")
# =============================================================
# 2. Paths
# =============================================================

DATA_ROOT = Path("/kaggle/input/datasets/social888/plantdisease/leaf disease.yolov8")
WORKDIR = Path("/kaggle/working")

META_REGISTRY = WORKDIR / "experiment_registry.json"
SPLIT_DIR = WORKDIR / "splits"
PRED_DIR = WORKDIR / "predictions"

for p in [SPLIT_DIR, PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# =============================================================
# 3. Reproducibility + memory optimization
# =============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def clear_memory(model=None):
    if model is not None:
        del model
    gc.collect()
    torch.cuda.empty_cache()

def hard_reset():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()


# =============================================================
# 4. Load metadata
# =============================================================

meta = pd.read_csv(DATA_ROOT / "metadata.csv")

meta["strata"] = (
    meta["crop"].astype(str) + "_" +
    meta["label"].astype(str) + "_" +
    meta["environment"].astype(str)
)


# =============================================================
# 5. Fixed Stratified Split (NO LEAKAGE)
# =============================================================

split1 = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=SEED)
train_val_idx, test_idx = next(split1.split(meta, meta["strata"]))

train_val = meta.iloc[train_val_idx]
test = meta.iloc[test_idx]

split2 = StratifiedShuffleSplit(n_splits=1, test_size=0.111, random_state=SEED)
train_idx, val_idx = next(split2.split(train_val, train_val["strata"]))

train_df = train_val.iloc[train_idx]
val_df = train_val.iloc[val_idx]
# =============================================================
# FIX: dataset safety filter (reduces corrupt logs + crashes)
# =============================================================
def valid_sample(img_name):
    img = DATA_ROOT / "train" / "images" / img_name
    lbl = DATA_ROOT / "train" / "labels" / img_name.replace(".jpg", ".txt")
    return img.exists() and lbl.exists()

train_df = train_df[train_df["image_name"].apply(valid_sample)]
val_df = val_df[val_df["image_name"].apply(valid_sample)]
test = test[test["image_name"].apply(valid_sample)]

# =============================================================
# 6. Dataset builder
# =============================================================
def copy_split(df, name):
    out = SPLIT_DIR / name

    (out / "images").mkdir(parents=True, exist_ok=True)
    (out / "labels").mkdir(parents=True, exist_ok=True)

    for img in df["image_name"]:

        shutil.copy(
            DATA_ROOT / "train" / "images" / img,
            out / "images" / img
        )

        lbl = img.replace(".jpg", ".txt")

        lp = DATA_ROOT / "train" / "labels" / lbl

        if lp.exists():
            shutil.copy(lp, out / "labels" / lbl)

copy_split(train_df, "train")
copy_split(val_df, "val")
copy_split(test, "test")

# =============================================================
# 7. YOLO config
# =============================================================

yaml_path = WORKDIR / "data.yaml"
yaml_path.write_text(f"""
path: {SPLIT_DIR}

train: train/images
val: val/images
test: test/images

nc: 9
names:
  0: maize fall armyworm
  1: maize leaf
  2: maize leaf blight
  3: potato late blight
  4: potato leaf
  5: potato pest damage
  6: tomato early blight
  7: tomato leaf
  8: tomato leaf miner
""")


# =============================================================
# 8. Split statistics logging
# =============================================================

def log_split_stats(df, name):
    return {
        "split": name,
        "size": len(df),
        "percent": len(df) / len(meta) * 100,
        "crop_distribution": df["crop"].value_counts(normalize=True).to_dict(),
        "label_distribution": df["label"].value_counts(normalize=True).to_dict(),
        "env_distribution": df["environment"].value_counts(normalize=True).to_dict(),
    }

split_stats = [
    log_split_stats(train_df, "train"),
    log_split_stats(val_df, "val"),
    log_split_stats(test, "test")
]

pd.DataFrame(split_stats).to_json(
    WORKDIR / "split_statistics.json",
    orient="records",
    indent=2
)

for s in split_stats:
    print("\n", s["split"])
    print("size:", s["size"])
    print("percent:", round(s["percent"], 2))
# =============================================================
# 8. Experiment Registry (PERSISTENCE)
# =============================================================

def save_registry(entry):
    data = []
    if META_REGISTRY.exists():
        data = json.loads(META_REGISTRY.read_text())
    data.append(entry)
    META_REGISTRY.write_text(json.dumps(data, indent=2))

# # =============================================================
# # 9. Severity computation (CORE GAP FIX)
# # =============================================================

# def compute_severity(mask_leaf, mask_disease):
#     leaf_area = mask_leaf.sum()
#     disease_area = mask_disease.sum()
#     return disease_area / (leaf_area + 1e-6)

# =============================================================
# 10. Evaluation utilities
# =============================================================

# def eval_by_domain(model, df, domain):
#     subset = df[df["environment"] == domain]

#     metrics = model.val()

#     return {
#         "seg_map50": metrics.seg.map50,
#         "seg_map5095": metrics.seg.map
#     }
def eval_by_domain(model, full_df, domain):

    subset = full_df[full_df["environment"] == domain]

    temp_dir = SPLIT_DIR / f"eval_{domain}"
    if temp_dir.exists():
        shutil.rmtree(temp_dir)

    images_dir = temp_dir / "images"
    labels_dir = temp_dir / "labels"

    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)

    # Copy only subset files
    for img_name in subset["image_name"]:

        src_img = DATA_ROOT / "train" / "images" / img_name
        dst_img = images_dir / img_name

        shutil.copy(src_img, dst_img)

        lbl_name = img_name.replace(".jpg", ".txt")

        src_lbl = DATA_ROOT / "train" / "labels" / lbl_name
        dst_lbl = labels_dir / lbl_name

        if src_lbl.exists():
            shutil.copy(src_lbl, dst_lbl)

    # Temporary yaml
    temp_yaml = WORKDIR / f"{domain}_eval.yaml"

    temp_yaml.write_text(f"""
path: {temp_dir}

train: images
val: images

nc: 9
names:
  0: maize fall armyworm
  1: maize leaf
  2: maize leaf blight
  3: potato late blight
  4: potato leaf
  5: potato pest damage
  6: tomato early blight
  7: tomato leaf
  8: tomato leaf miner
""")

    metrics = model.val(
        data=str(temp_yaml),
        split="val",
        batch=16,
        workers=2,
        verbose=False
    )

    return {
        "seg_map50": metrics.seg.map50,
        "seg_map5095": metrics.seg.map
    }
# =============================================================
# 11. Prediction logging (CORE GAP FIX)
# =============================================================

# def log_predictions(model, images, run_name):
#     out_dir = PRED_DIR / run_name
#     out_dir.mkdir(exist_ok=True, parents=True)

#     for img in images[:20]:
#         r = model.predict(str(DATA_ROOT / "train" / "images" / img))
#         r[0].save(out_dir / img)

# =============================================================
# 12. Training function (RESUME + ABLATION CONTROL)
# =============================================================

# =============================================================
# 12. Training function (RESUME TRAINING SUPPORT)
# =============================================================

def train(
    model_name,
    img_size,
    exp_name,
    extra=None,
    resume=False,
    epochs=80,
    batch=16
):

    model = YOLO(model_name)

    with mlflow.start_run(run_name=exp_name):

        results = model.train(

        data=str(yaml_path),
    
        imgsz=img_size,
    
        epochs=epochs,
    
        batch=batch,
    
        workers=2,
    
        cache=False,
    
        patience=15,
    
        resume=resume,
    
        device=0,
    
        verbose=False,
    
        **(extra or {})
    )
        # =====================================================
        # TEST EVALUATION
        # =====================================================

        metrics = model.val(

            data=str(yaml_path),

            split="test",

            batch=batch,

            workers=2,

            verbose=False
        )

        # =====================================================
        # DOMAIN EVALUATION
        # =====================================================

        field_metrics = eval_by_domain(model, test, "field")

        lab_metrics = eval_by_domain(model, test, "lab")

        # =====================================================
        # PRINT RESULTS
        # =====================================================

        print(f"\n=== {exp_name} RESULTS ===")

        print(f"Seg mAP50: {metrics.seg.map50:.4f}")

        print(f"Seg mAP50-95: {metrics.seg.map:.4f}")

        print(f"Box mAP50: {metrics.box.map50:.4f}")

        print(f"Box mAP50-95: {metrics.box.map:.4f}")

        print(f"Precision: {metrics.seg.mp:.4f}")

        print(f"Recall: {metrics.seg.mr:.4f}")

        print(f"Field Seg mAP50: {field_metrics['seg_map50']:.4f}")

        print(f"Lab Seg mAP50: {lab_metrics['seg_map50']:.4f}")

        # =====================================================
        # MLFLOW LOGGING
        # =====================================================

        mlflow.log_param("model", model_name)

        mlflow.log_param("img_size", img_size)

        mlflow.log_param("epochs", epochs)

        mlflow.log_param("batch", batch)

        mlflow.log_param("resume", resume)

        # Main segmentation metrics
        mlflow.log_metric("seg_mAP50", metrics.seg.map50)

        mlflow.log_metric("seg_mAP50_95", metrics.seg.map)

        # Box metrics
        mlflow.log_metric("mAP50", metrics.box.map50)

        mlflow.log_metric("mAP50_95", metrics.box.map)

        # Domain metrics
        mlflow.log_metric(
            "field_seg_mAP50",
            field_metrics["seg_map50"]
        )

        mlflow.log_metric(
            "lab_seg_mAP50",
            lab_metrics["seg_map50"]
        )

        mlflow.log_metric(
            "field_seg_mAP50_95",
            field_metrics["seg_map5095"]
        )

        mlflow.log_metric(
            "lab_seg_mAP50_95",
            lab_metrics["seg_map5095"]
        )

        # Precision / Recall
        mlflow.log_metric("seg_precision", metrics.seg.mp)

        mlflow.log_metric("seg_recall", metrics.seg.mr)

        # =====================================================
        # SAVE REGISTRY
        # =====================================================

        save_registry({

            "exp": exp_name,

            "model": model_name,

            "imgsz": img_size,

            "epochs": epochs,

            "batch": batch,

            "resume": resume,

            "seg_mAP50": float(metrics.seg.map50),

            "seg_mAP50_95": float(metrics.seg.map),

            "box_mAP50": float(metrics.box.map50),

            "seg_precision": float(metrics.seg.mp),

            "seg_recall": float(metrics.seg.mr),

            "field_seg_mAP50": float(
                field_metrics["seg_map50"]
            ),

            "lab_seg_mAP50": float(
                lab_metrics["seg_map50"]
            ),

            "field_seg_mAP50_95": float(
                field_metrics["seg_map5095"]
            ),

            "lab_seg_mAP50_95": float(
                lab_metrics["seg_map5095"]
            )
        })

    clear_memory(model)

    hard_reset()

    return model, metrics



# =============================================================
# CONTINUE TRAINING FROM last.pt
# =============================================================

LAST_PT = "/kaggle/input/models/social888/last/pytorch/default/1/last.pt"

aug = dict(
    hsv_h=0.015,
    hsv_s=0.30,
    hsv_v=0.20,

    degrees=5,
    translate=0.03,
    scale=0.20,

    fliplr=0.5,
    flipud=0.0,

    mosaic=0.4,
    mixup=0.05,
    copy_paste=0.15,
    erasing=0.1,

    close_mosaic=20
)

m10, r10 = train(

    model_name=LAST_PT,

    img_size=512,

    exp_name="nano_better_aug_512_resume80",

    extra=aug,

    resume=True,

    epochs=80,

    batch=16
)
# =============================================================
# 17. MOSAIC SCHEDULING FIX (POST PROCESS IDEA)
# =============================================================

# YOLO already handles close_mosaic; ensure final fine-tuning phase
# would be:
# epochs_split = [0-45 mosaic on, 45-60 off]

# # =============================================================
# # 18. VISUALIZATION & REPORTING (LOW SEVERITY FIX)
# # =============================================================

# import json

# with open(META_REGISTRY, "r") as f:
#     data = json.load(f)

# history = pd.DataFrame(data)

# plt.plot(history["seg_mAP50"])
# plt.title("mAP50 Across Experiments")
# plt.show()

# # =============================================================
# # 19. FINAL OUTPUT
# # =============================================================

# print("Final model:", BEST)


# =============================================================
# 20. FINAL SUMMARY EXPORT (ADD HERE)
# =============================================================

import json

df = pd.DataFrame(json.loads(META_REGISTRY.read_text()))
df.to_csv(WORKDIR / "all_experiments_summary.csv", index=False)

print("\nSaved experiment summary to:")
print(WORKDIR / "all_experiments_summary.csv")